# Notebook 04 – Feature Importance (Random Forest + SHAP)

Identify which user context variables most influence UI preferences using grouped predictive modelling:

- **Global UI**
- **Desktop UI**
- **Mobile UI**

Each group is processed automatically in a single pipeline loop.


## Expected Outputs
- Consolidated: `feature_importance`, `shap_summary`, `model_metrics`, `target_predictor_summary`
- Group reports: `group_reports/global_ui_summary`, `desktop_ui_summary`, `mobile_ui_summary`
- Plots: `plots/Global_UI/`, `plots/Desktop_UI/`, `plots/Mobile_UI/`
- `pipeline_summary.md`


In [8]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


_bootstrap_project()

from src.utils.dependencies import ensure_packages

ensure_packages("shap", "scikit-learn")

from src.machine_learning.pipeline import run_feature_importance_pipeline
from src.preprocessing.columns import get_ml_feature_columns, get_ui_target_groups
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)
plt.rcParams["figure.dpi"] = 300

PATHS, REPORTS = setup_notebook("Feature_Importance")
feature_columns = get_ml_feature_columns()
ui_target_groups = get_ui_target_groups()

for group_name, targets in ui_target_groups.items():
    print(f"{group_name}: {len(targets)} targets")



Global_UI: 14 targets
Desktop_UI: 13 targets
Mobile_UI: 14 targets


## Load Data


In [9]:
df = pd.read_csv(PATHS.data_processed / "clean_dataset.csv")
statistical_results = pd.read_csv(
    PATHS.reports / "Statistical_Validation" / "tables" / "statistical_results.csv"
)
logger.info("Loaded clean dataset: %s rows", len(df))
display(df[feature_columns].head())


INFO: Loaded clean dataset: 200 rows


,primary_persona,current_mood,primary_device,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness
0,The Impulsive Buyer (I make quick decisions ba...,Neutral,Smartphone,3.0,3.0,3.0,3.0,3.0
1,The Loyal Customer (I stick with brands and st...,Happy,Smartphone,3.5,4.5,3.5,2.5,2.5
2,"The Minimalist (I want simple, efficient shopp...",Happy,Smartphone,4.0,3.0,3.0,3.5,3.0
3,The Researcher (I thoroughly research products...,Bored,Smartphone,2.5,5.0,4.0,2.0,2.5
4,The Impulsive Buyer (I make quick decisions ba...,Neutral,Smartphone,4.0,5.0,2.5,1.0,4.0


## Run Grouped Feature Importance Pipeline


In [10]:
pipeline_result = run_feature_importance_pipeline(
    df=df,
    feature_columns=feature_columns,
    reports_dir=REPORTS,
    ui_target_groups=ui_target_groups,
)

model_metrics = pipeline_result.model_metrics
feature_importance = pipeline_result.feature_importance
shap_summary = pipeline_result.shap_summary
target_summary = pipeline_result.target_summary
overall_importance = pipeline_result.overall_importance
group_importance = pipeline_result.group_importance

print(f"Models trained: {len(model_metrics)}")
display(model_metrics.groupby('UI_Group')['accuracy'].mean())


INFO: Processing UI group: Global_UI (14 targets)


INFO: Trained font_style_pref | accuracy=0.225 | best={'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
INFO: Trained font_size_pref | accuracy=0.400 | best={'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
INFO: Trained color_theme_pref | accuracy=0.275 | best={'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
/Users/mariam/Library/Python/3.9/lib/python/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
INFO: Trained accent_color_pref | accuracy=0.550 | best={'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
INFO: Trained background_pref | accuracy=0.275 | best={'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples

Models trained: 41


UI_Group
Desktop_UI    0.438462
Global_UI     0.378571
Mobile_UI     0.517857
Name: accuracy, dtype: float64

<Figure size 1920x1440 with 0 Axes>

...

<Figure size 1920x1440 with 0 Axes>

<Figure size 1920x1440 with 0 Axes>

<Figure size 1920x1440 with 0 Axes>

<Figure size 1920x1440 with 0 Axes>

<Figure size 1920x1440 with 0 Axes>

## Consolidated Results Preview


In [11]:
display(feature_importance.groupby('UI_Group').head(3))
display(target_summary.head(10))
display(group_importance)


,UI_Group,UI_Target,Predictor,Importance,Rank
0,Global_UI,font_style_pref,Neuroticism,0.166754,1
1,Global_UI,font_style_pref,Openness,0.150461,2
2,Global_UI,font_style_pref,Conscientiousness,0.138570,3
112,Desktop_UI,desktop_grid_pref,Agreeableness,0.148965,1
113,Desktop_UI,desktop_grid_pref,Conscientiousness,0.147140,2
114,Desktop_UI,desktop_grid_pref,Neuroticism,0.139237,3
216,Mobile_UI,mobile_grid_pref,Neuroticism,0.154625,1
217,Mobile_UI,mobile_grid_pref,Openness,0.146460,2
218,Mobile_UI,mobile_grid_pref,Conscientiousness,0.143302,3


,UI_Group,UI_Target,Top_Predictor,Top_Predictor_Importance,Top_3_Predictors,Top_3_Importance_Values,Top_SHAP_Predictors,Top_SHAP_Values,Top_Mean_ABS_SHAP
0,Global_UI,font_style_pref,Neuroticism,0.166754,"Neuroticism, Openness, Conscientiousness","0.1668, 0.1505, 0.1386","primary_device, Neuroticism, current_mood","0.0483, 0.0418, 0.0311",0.048283
1,Global_UI,font_size_pref,Neuroticism,0.175039,"Neuroticism, Extraversion, Agreeableness","0.1750, 0.1392, 0.1336","current_mood, primary_persona, Extraversion","0.0462, 0.0355, 0.0321",0.046156
2,Global_UI,color_theme_pref,primary_persona,0.161543,"primary_persona, Neuroticism, current_mood","0.1615, 0.1456, 0.1414","primary_persona, Agreeableness, Neuroticism","0.0297, 0.0236, 0.0201",0.029735
3,Global_UI,accent_color_pref,current_mood,0.146994,"current_mood, Openness, Conscientiousness","0.1470, 0.1450, 0.1422","Openness, Neuroticism, current_mood","0.0274, 0.0197, 0.0191",0.027365
4,Global_UI,background_pref,Neuroticism,0.153902,"Neuroticism, Conscientiousness, primary_persona","0.1539, 0.1407, 0.1361","current_mood, Neuroticism, Extraversion","0.0483, 0.0318, 0.0272",0.048290
5,Global_UI,whitespace_pref,current_mood,0.150778,"current_mood, Neuroticism, Extraversion","0.1508, 0.1506, 0.1440","Openness, Neuroticism, Conscientiousness","0.0339, 0.0300, 0.0261",0.033889
6,Global_UI,button_style_pref,Neuroticism,0.166657,"Neuroticism, Conscientiousness, Extraversion","0.1667, 0.1406, 0.1381","Agreeableness, Openness, current_mood","0.0250, 0.0244, 0.0219",0.025045
7,Global_UI,hero_banner_size,Neuroticism,0.160897,"Neuroticism, Openness, Extraversion","0.1609, 0.1453, 0.1421","current_mood, Openness, primary_device","0.0408, 0.0405, 0.0306",0.040788
8,Global_UI,recommendation_type,Neuroticism,0.154047,"Neuroticism, Agreeableness, Extraversion","0.1540, 0.1435, 0.1403","Neuroticism, primary_persona, current_mood","0.0406, 0.0362, 0.0311",0.040556
9,Global_UI,social_proof_display,Neuroticism,0.160564,"Neuroticism, Conscientiousness, primary_persona","0.1606, 0.1469, 0.1468","Neuroticism, Extraversion, primary_device","0.0315, 0.0252, 0.0230",0.031520


,UI_Group,Predictor,Importance
3,Desktop_UI,Neuroticism,0.156502
0,Desktop_UI,Agreeableness,0.139934
1,Desktop_UI,Conscientiousness,0.138600
4,Desktop_UI,Openness,0.133618
2,Desktop_UI,Extraversion,0.133580
7,Desktop_UI,primary_persona,0.130799
5,Desktop_UI,current_mood,0.130176
6,Desktop_UI,primary_device,0.036791
11,Global_UI,Neuroticism,0.155291
15,Global_UI,primary_persona,0.139949


## Compare with Notebook 03 (Reference Only)


In [12]:
comparison = statistical_results[[
    'Predictor', 'UI_Element', 'Raw_P', 'Cramers_V', 'Evidence_Score'
]].rename(columns={'UI_Element': 'UI_Target', 'Predictor': 'Statistical_Predictor'})

comparison_preview = target_summary.merge(
    comparison,
    on='UI_Target',
    how='left',
)
display(comparison_preview.head(10))


,UI_Group,UI_Target,Top_Predictor,Top_Predictor_Importance,Top_3_Predictors,Top_3_Importance_Values,Top_SHAP_Predictors,Top_SHAP_Values,Top_Mean_ABS_SHAP,Statistical_Predictor,Raw_P,Cramers_V,Evidence_Score
0,Global_UI,font_style_pref,Neuroticism,0.166754,"Neuroticism, Openness, Conscientiousness","0.1668, 0.1505, 0.1386","primary_device, Neuroticism, current_mood","0.0483, 0.0418, 0.0311",0.048283,primary_persona,0.166999,0.183167,0.665831
1,Global_UI,font_style_pref,Neuroticism,0.166754,"Neuroticism, Openness, Conscientiousness","0.1668, 0.1505, 0.1386","primary_device, Neuroticism, current_mood","0.0483, 0.0418, 0.0311",0.048283,current_mood,0.648708,0.173228,0.416997
2,Global_UI,font_style_pref,Neuroticism,0.166754,"Neuroticism, Openness, Conscientiousness","0.1668, 0.1505, 0.1386","primary_device, Neuroticism, current_mood","0.0483, 0.0418, 0.0311",0.048283,Openness_Level,0.078484,0.168357,0.709963
3,Global_UI,font_style_pref,Neuroticism,0.166754,"Neuroticism, Openness, Conscientiousness","0.1668, 0.1505, 0.1386","primary_device, Neuroticism, current_mood","0.0483, 0.0418, 0.0311",0.048283,Conscientiousness_Level,0.321949,0.132175,0.517859
4,Global_UI,font_style_pref,Neuroticism,0.166754,"Neuroticism, Openness, Conscientiousness","0.1668, 0.1505, 0.1386","primary_device, Neuroticism, current_mood","0.0483, 0.0418, 0.0311",0.048283,Agreeableness_Level,0.444352,0.120561,0.423491
5,Global_UI,font_style_pref,Neuroticism,0.166754,"Neuroticism, Openness, Conscientiousness","0.1668, 0.1505, 0.1386","primary_device, Neuroticism, current_mood","0.0483, 0.0418, 0.0311",0.048283,primary_device,0.406797,0.120482,0.378340
6,Global_UI,font_style_pref,Neuroticism,0.166754,"Neuroticism, Openness, Conscientiousness","0.1668, 0.1505, 0.1386","primary_device, Neuroticism, current_mood","0.0483, 0.0418, 0.0311",0.048283,Neuroticism_Level,0.589414,0.107820,0.322744
7,Global_UI,font_style_pref,Neuroticism,0.166754,"Neuroticism, Openness, Conscientiousness","0.1668, 0.1505, 0.1386","primary_device, Neuroticism, current_mood","0.0483, 0.0418, 0.0311",0.048283,Extraversion_Level,0.993681,0.042895,0.063733
8,Global_UI,font_size_pref,Neuroticism,0.175039,"Neuroticism, Extraversion, Agreeableness","0.1750, 0.1392, 0.1336","current_mood, primary_persona, Extraversion","0.0462, 0.0355, 0.0321",0.046156,current_mood,0.710478,0.168376,0.380479
9,Global_UI,font_size_pref,Neuroticism,0.175039,"Neuroticism, Extraversion, Agreeableness","0.1750, 0.1392, 0.1336","current_mood, primary_persona, Extraversion","0.0462, 0.0355, 0.0321",0.046156,Conscientiousness_Level,0.122491,0.158522,0.690442


## Final Summary


In [13]:
best_model = model_metrics.loc[model_metrics['accuracy'].idxmax()]
avg_accuracy = model_metrics['accuracy'].mean()

print(f"Best performing UI prediction: {best_model['UI_Target']} ({best_model['UI_Group']}, accuracy={best_model['accuracy']:.3f})")
print(f"Average model accuracy: {avg_accuracy:.3f}")
print("Top 20 most influential predictors overall:")
print(overall_importance.to_string(index=False))
print("\nExport locations:")
for name, path in pipeline_result.export_paths.items():
    print(f"- {name}: {path}")


Best performing UI prediction: mobile_whitespace (Mobile_UI, accuracy=0.850)
Average model accuracy: 0.445
Top 20 most influential predictors overall:
        Predictor  Importance
      Neuroticism    0.155312
Conscientiousness    0.140542
         Openness    0.136858
    Agreeableness    0.134741
  primary_persona    0.134502
     Extraversion    0.132684
     current_mood    0.129806
   primary_device    0.035554

Export locations:
- csv: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Feature_Importance/group_importance.csv
- xlsx: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Feature_Importance/group_importance.xlsx
- pipeline_summary_md: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Feature_Importance/pipeline_summary.md
